In [215]:
'''

Data: Rides data in San Francisco between the period 2023-01-01 AND 2026-07-31.
This is dummy data generated using Gemini. 
 
Rides Data: The data is in "sf_cab_requests_2023_2026_1M.csv" file
 
Zone type data: We gathered information on different zones in San Francisco from "https://data.sfgov.org/api/geospatial/3i4a-hu95?method=export&format=GeoJSON"

Weather Data: We can gathered hourly weather data from "https://open-meteo.com/"

Using Uber's Geo Spatial indexing system, we divided the City into Hexagons of roughly 2sqmiles each.
We used Resolution = 7.


What does the script do?
- The script analyzes Rides data, answers a few business questions.
- It predicts the demand for cabs across different Hexagons in the SF over the next 1 hour

 Note:
This script is a mix of hand written code and a few snippets generated using Gemini
 
 
'''

'\n\nData: Rides data in San Francisco between the period 2023-01-01 AND 2026-07-31.\nThis is dummy data generated using Gemini. \n \nRides Data: The data is in "sf_cab_requests_2023_2026_1M.csv" file\n \nZone type data: We gathered information on different zones in San Francisco from "https://data.sfgov.org/api/geospatial/3i4a-hu95?method=export&format=GeoJSON"\n\nWeather Data: We can gathered hourly weather data from "https://open-meteo.com/"\n\nUsing Uber\'s Geo Spatial indexing system, we divided the City into Hexagons of roughly 2sqmiles each.\nWe used Resolution = 7.\n\n\nWhat does the script do?\n- The script analyzes Rides data, answers a few business questions.\n- It predicts the demand for cabs across different Hexagons in the city over the next 1 hour\n\n Note:\nThis script is a mix of hand written code and a few snippets generated using Gemini\n \n \n'

In [12]:
###########################################  Load the necessary libraries  ############################################

In [127]:
import pandas as pd
import geopandas as gpd

import numpy as nm

import shapely as sh
from shapely import geometry
from shapely.geometry import Point
from shapely.geometry import Polygon

import h3
import folium

import json

import requests

from datetime import datetime, timedelta

In [2]:
###############################################  Load Rides Data  #################################################

In [ ]:
#Load rides data from a CSV into a DataFrame

#Note: The Data set used is dummy data generated using Gemini. 
#The patterns found in the data may not represent the Cab industry accurately

In [3]:
rides_data = pd.read_csv("sf_cab_requests_2023_2026_1M.csv")

In [ ]:
##############################################  Add calculated fields  ############################################

In [ ]:
############### Calculate Ride type using Ride Distance

In [14]:
rides_data = rides_data.drop("rides_type", axis=1, errors="ignore")

conditions = [
    (rides_data["distance_travelled_meters"] <= 1610),   #<=1mile
    (rides_data["distance_travelled_meters"] > 1610) & (rides_data["distance_travelled_meters"] <= 8047),   #>1mile - 5miles
    (rides_data["distance_travelled_meters"] > 8047) & (rides_data["distance_travelled_meters"] <= 40234), #>5miles - 25miles
    (rides_data["distance_travelled_meters"] > 40234) & (rides_data["distance_travelled_meters"] <= 96561), #>25miles - 60miles
    (rides_data["distance_travelled_meters"] > 96561)  #>60miles    
]

choices = ["Neighborhood Rides", "Short Rides", "Medium Rides", "Long Distance Rides", "Long Haul Rides"]

rides_data["rides_type"] = nm.select(conditions, choices, default = "UNK")

In [ ]:
############### Calculate Ride Year, Month, Day, Week etc. from Request Datetime

In [58]:
rides_data["request_datetime"] = pd.to_datetime(rides_data["request_datetime"])  #Convert the column into Datetime format


rides_data["ride_year"] = rides_data["request_datetime"].dt.year

rides_data["request_date"] = rides_data["request_datetime"].dt.date

rides_data["request_hour"] = rides_data["request_datetime"].dt.hour

rides_data["request_day"] = rides_data["request_datetime"].dt.day

rides_data["request_week_of_month"] = (rides_data["request_datetime"].dt.day - 1) // 7 + 1

rides_data["request_month"] = rides_data["request_datetime"].dt.month

rides_data["day_name"] = None

l = len(rides_data)

for i in range(l):
    rides_data["day_name"][i] = rides_data["request_datetime"][i].strftime('%A')


/var/folders/rx/py7wx93s12d9vv_55b6msmvc0000gn/T/ipykernel_37874/2658142530.py:21: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  rides_data["day_name"][i] = rides_data["request_datetime"][i].strftime('%A')
/var/folders/rx/py7wx93s12d9vv_55b6

In [ ]:
############### Calculate Price per meter

In [16]:
rides_data["price_per_mtr"] = rides_data["ride_cost"]/ rides_data["distance_travelled_meters"]

In [ ]:
############### Calculate Ride type using Ride Distance

In [17]:
rides_data = rides_data.drop("rides_type", axis=1, errors="ignore")

conditions = [
    (rides_data["distance_travelled_meters"] <= 1610),   #<=1mile
    (rides_data["distance_travelled_meters"] > 1610) & (rides_data["distance_travelled_meters"] <= 8047),   #>1mile - 5miles
    (rides_data["distance_travelled_meters"] > 8047) & (rides_data["distance_travelled_meters"] <= 40234), #>5miles - 25miles
    (rides_data["distance_travelled_meters"] > 40234) & (rides_data["distance_travelled_meters"] <= 96561), #>25miles - 60miles
    (rides_data["distance_travelled_meters"] > 96561)  #>60miles    
]

choices = ["Neighborhood Rides", "Short Rides", "Medium Rides", "Long Distance Rides", "Long Haul Rides"]

rides_data["rides_type"] = nm.select(conditions, choices, default = "UNK")

In [ ]:
############### Calculate Day Segment

In [ ]:
#Let us divide the day into few segments 
'''
Uber's dynamic pricing algorithm and driver-facing "Hourly Trends" tools segment the day into specific 
behavioral blocks to balance the supply of drivers with rider demand:


Early Morning (4:00 AM – 7:00 AM): Characterized as low-volume but high-efficiency, 
dominated by airport departures and business travel.


Morning Commute (7:00 AM – 10:00 AM): A weekday peak segment featuring high-frequency, 
predictable commuter trips into commercial districts.


Mid-day Off-Peak (10:00 AM – 4:00 PM): The baseline rate segment (typically Monday through Wednesday), 
yielding the lowest prices for riders due to balanced supply.


Evening Rush / Social Peak (4:00 PM – 8:00 PM): High demand stemming from combined workforce departures, 
happy hours, and social dinner trips.


Nightlife / Weekend Late Night (10:00 PM – 2:00 AM): The highest premium segment(concentrated on Thursday–Saturday),
heavily prone to surge pricing multipliers as bar and event crowds disperse.


Weekend Brunch Peak (10:00 AM – 2:00 PM): A unique weekend segment (primarily Sundays) that experiences a 
localized surge independent of typical weekday commute rules.
'''

In [18]:
#Day Segments

EM_LL = datetime.strptime("04:00:00", "%H:%M:%S").time()
EM_HL = datetime.strptime("07:00:00", "%H:%M:%S").time()


MC_LL = datetime.strptime("07:00:01", "%H:%M:%S").time()
MC_HL = datetime.strptime("10:00:00", "%H:%M:%S").time()


MID_LL = datetime.strptime("10:00:01", "%H:%M:%S").time()
MID_HL = datetime.strptime("16:00:00", "%H:%M:%S").time()


ER_LL = datetime.strptime("16:00:01", "%H:%M:%S").time()
ER_HL = datetime.strptime("20:00:00", "%H:%M:%S").time()


NL_LL = datetime.strptime("22:00:00", "%H:%M:%S").time()
NL_HL = datetime.strptime("02:00:00", "%H:%M:%S").time()



In [20]:
l = len(rides_data)

rides_data["day_segment"] = None

for i in range(l):
    if (rides_data["request_datetime"][i].time() > EM_LL) & (rides_data["request_datetime"][i].time() < EM_HL):
        rides_data["day_segment"][i] = 'Early Morning'
        
    elif (rides_data["request_datetime"][i].time() > MC_LL) & (rides_data["request_datetime"][i].time() < MC_HL):
        rides_data["day_segment"][i] = 'Morning Commute'
        
    elif (rides_data["request_datetime"][i].time() > MID_LL) & (rides_data["request_datetime"][i].time() < MID_HL):
        rides_data["day_segment"][i] = 'Mid-day Off-Peak'
        
    elif (rides_data["request_datetime"][i].time() > ER_LL) & (rides_data["request_datetime"][i].time() < ER_HL):
        rides_data["day_segment"][i] = 'Evening Rush'
        
    elif (rides_data["request_datetime"][i].time() > NL_LL) & (rides_data["request_datetime"][i].time() < NL_LL):
        rides_data["day_segment"][i] = 'Nightlife'

    else: rides_data["day_segment"][i] = 'Other'


IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [9]:
#########################################  Add H3 index to Rides Data  #########################################

In [10]:
#Adding H3 INDEX to the rides data using Pick up Latitude and Longitude and Drop off Latitude and Longitude

#Let us add the H3 Indexc to the data. Area of each hex = 2sqmiles. 
#Resolution = 7

In [11]:
#Index coordinates to Hexagons

#Let us keeop the Hexagons to 2sq miles in area to capture neighborhood level demand
#Using the Uber's H3 Resolution formula, the resolution is 7


h3_resolution = 7
rides_data["pickup_h3_index"] = None
rides_data["dropoff_h3_index"] = None


for i in range(len(rides_data)):
    rides_data["pickup_h3_index"][i] = h3.latlng_to_cell(rides_data["pickup_latitude"][i], rides_data["pickup_longitude"][i], res = h3_resolution)
    rides_data["dropoff_h3_index"][i] = h3.latlng_to_cell(rides_data["dropoff_latitude"][i], rides_data["dropoff_longitude"][i], res = h3_resolution)
    

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [13]:
#Create a unique list of h3_indexes for fetching data from external APIs

pickup_locn_uniq = pd.DataFrame(columns = ['h3_index'])

pickup_locn_uniq['h3_index'] = pd.DataFrame(rides_data["pickup_h3_index"].unique())

pickup_locn_uniq['hex_center_lat'] = None
pickup_locn_uniq['hex_center_lng'] = None

print(pickup_locn_uniq["h3_index"].nunique())

86


In [ ]:
#############################################  Add zone Data  ################################################  

In [21]:
#Zoning the city - Adding Zone Category types to the City
#Data: https://data.sfgov.org/api/geospatial/3i4a-hu95?method=export&format=GeoJSON

In [22]:
# Open the file and load its contents
with open('Zoning Map - Zoning Districts.geojson', 'r', encoding='utf-8') as file:
    zone_data_json = json.load(file)

In [23]:
#Inspect Top-Level Keys
print("Top-level GeoJSON keys:", list(zone_data_json.keys()))


#Inspect Property Keys (Columns)
sample_properties = zone_data_json["features"][0]["properties"]
print("Attribute Keys / Columns:", list(sample_properties.keys()))


#Convert to a Pandas DataFrame
# pd.json_normalize flattens the list of feature dictionaries
zone_data = pd.json_normalize(zone_data_json["features"])

Top-level GeoJSON keys: ['type', 'features']
Attribute Keys / Columns: ['zoning_sim', 'districtna', 'url', 'zoning', 'codesectio', 'gen']


In [24]:
'''
We have h3_index and polygon shape boundary coordinates. Using this information we will check if the 
pick up coordinates fall in this shape and then attribute the corresponding zone type.

'''

'\nWe have h3_index and polygon shape boundary coordinates. Using this information we will check if the \npick up coordinates fall in this shape and then attribute the corresponding zone type.\n\n'

In [25]:
'''
To join your point/H3 dataset with the Zoning MultiPolygon dataset, use a spatial join (gpd.sjoin) in GeoPandas. 
This matches each coordinate point inside the polygon it falls within.
'''

#The code below in this cell has been generated using Gemini:

# ==============================================================================
# 1. LOAD YOUR DATA (Points + H3 Index)
# ==============================================================================

# Convert lat/lng to Shapely Point objects: Point(Longitude, Latitude)
points_geometry = [Point(xy) for xy in zip(rides_data["pickup_longitude"], rides_data["pickup_latitude"])]

# Create a GeoDataFrame for your points with WGS84 CRS (EPSG:4326)
points_gdf = gpd.GeoDataFrame(rides_data, geometry=points_geometry, crs="EPSG:4326")

# ==============================================================================
# 2. LOAD THE SF ZONING MULTIPOLYGON GEOJSON
# ==============================================================================
# If loading from local file: sf_zoning_gdf = gpd.read_file("Zoning Map - Zoning Districts.geojson")
zoning_source = "https://data.sfgov.org/api/geospatial/3i4a-hu95?method=export&format=GeoJSON"
sf_zoning_gdf = gpd.read_file(zoning_source)

# Ensure the zoning GeoDataFrame uses the same coordinate system (EPSG:4326)
if sf_zoning_gdf.crs != points_gdf.crs:
    sf_zoning_gdf = sf_zoning_gdf.to_crs(points_gdf.crs)

# ==============================================================================
# 3. PERFORM THE SPATIAL JOIN
# ==============================================================================
# predicate="within" checks if the Point is inside a Zoning MultiPolygon.
# how="left" retains all user points, even if a point falls into water/no zone.
rides_data_with_zone = gpd.sjoin(
    points_gdf,
    sf_zoning_gdf,
    how="left",
    predicate="within",
)

# ==============================================================================
# 4. CLEAN AND INSPECT THE FINAL DATAFRAME
# ==============================================================================
# Drop the internal spatial index column added by sjoin
if "index_right" in rides_data_with_zone.columns:
    rides_data_with_zone = rides_data_with_zone.drop(columns=["index_right"])


# Export to CSV / JSON
# final_df.to_csv("points_with_zoning.csv", index=False)

In [50]:
rides_data = rides_data_with_zone

In [ ]:
##############################################  Add weather Data  #############################################  

In [26]:
#Weather Data - Adding Weather Data to the rides data
#Weather Data Sourced from https://open-meteo.com/

In [51]:
################## Run this cell to load extracted weather data
# If you want to extract latest data then ignore this cell and proceed to next cell
weather_data = pd.read_csv("weather_data.csv")

In [ ]:
################## Extract weather data

import requests   #send http requests and intereact with web APIs

url = "https://archive-api.open-meteo.com/v1/archive"

ln = len(pickup_locn_uniq)

weather_data = pd.DataFrame()
weather_data = weather_data.iloc[0:0]

for i in range(ln):
    h3_index = pickup_locn_uniq["h3_index"][i]
    print("h3_index = ", h3_index, "    i = ", i)
    lat , lng = h3.cell_to_latlng(h3_index)
    
    
    params = {
        "latitude": lat,
        "longitude": lng,
        "start_date": '2023-01-01',
        "end_date": '2026-08-01',
        "hourly": "precipitation,temperature_2m,weather_code",
        "timezone": "America/Los_Angeles"
    }
    
    response = requests.get(url, params = params)
    weather_data_json = response.json()

    #Reach inside the JSON Dictionary and fetch the hourly dictionary
    weather_data_dict = weather_data_json["hourly"]

    #Convert the dictionay into a Data Frame
    weather_data_each_h3index = pd.DataFrame(weather_data_dict)
    weather_data_each_h3index["h3_index"] = h3_index
    weather_data = pd.concat([weather_data, weather_data_each_h3index], ignore_index = True)

    #Check if data is loaded into the Data Frame
    print(weather_data_each_h3index.shape)
    
    

#exporting weather data to a csv file for later use
weather_data.to_csv("weather_data.csv", index = False)

In [52]:
#convert time into YYYY-MM-DD HH:MM:SS
weather_data["time2"] = pd.to_datetime(weather_data["time"]).dt.strftime("%Y-%m-%d %H:%M:%S")
weather_data['time2'] = pd.to_datetime(weather_data['time2'])

In [53]:
############### Add weather code description
'''
The weather data above captures codes for different weathers. 
We will be using wmo_weather_codes.csv file to extract weather code description

'''

weather_code_def = pd.read_csv("wmo_weather_codes.csv")

weather_data = pd.merge(
    weather_data,
    weather_code_def[["weather_code", "condition"]],
    left_on = "weather_code",
    right_on = "weather_code",
    how = "left"
)

In [54]:
############# Join weather data to rides_data

#extract date & hour from the request datetime column and combine them back to YYYY-MM-DD HH:00:00 to join with weather data
rides_data["request_date"] = rides_data["request_datetime"].dt.date
rides_data["request_hour"] = rides_data["request_datetime"].dt.hour

#Combine date and hour into a datetime column as mentioned above
rides_data['predicted_datehour'] = pd.to_datetime(rides_data['request_date']) + pd.to_timedelta(rides_data['request_hour'], unit='h')  


rides_data = pd.merge(
    rides_data,
    weather_data,
    left_on = ["predicted_datehour", "pickup_h3_index"],
    right_on = ["time2", "h3_index"] ,
    how = 'left',
    suffixes = ("", "_wthr")

)

In [ ]:
#################################################  EDA  #####################################################  

In [55]:
#Metric Calculation

revenue_across_years = rides_data.groupby("ride_year").agg(
Total_Revenue = ("ride_cost", "sum"),
No_of_Rides = ("request_id", "count"),
No_of_Drivers = ("driver_id", "nunique"),
No_of_Riders = ("rider_id", "nunique")
)

by_ride_type = rides_data.groupby(["ride_year","rides_type"]).aggregate(
Revenue = ("ride_cost", "sum"),
No_of_Rides = ("request_id", "count")
)

by_cab_type = rides_data.groupby(["ride_year","requested_cab_type"]).aggregate(
Revenue = ("ride_cost", "sum"),
No_of_Rides = ("request_id", "count")
)


In [ ]:
#################################################  Forecasting rides  #####################################################  

In [ ]:
'''
Scenario: 
I am currently at hour T and I would like to predict the demand between T and T+1

Model Data:
Data Level: hex_id + hour


Y - Variable = No of rides requested (T to T+1)


X variables:

Recent Demand:
- Number of cabs booked between T-1 and T
- Number of cabs booked between T-2 and T-1
- Average number of cabs booked between T-5 TO T-2
- Price per mile for rides between T-1 and T
- Price per mile for rides between T-2 and T-1
- Average Price per mile for rides between T-5 and T-2


Same time in the past:
- Number of cabs booked between T and T+1 yesterday
- Average number of cabs booked last 3 days between T and T+1 yesterday
- Number of cabs booked between T and T+1 same day last week
- Average number of cabs booked between T and T+1 over the last 3 weeks
- Number of cabs booked between T and T+1 same day last month
- Number of cabs booked between T and T+1 same day last year
- Average Price per mile for rides between T and T+1 yesterday
- Average Price per mile for rides over last 3 days between T and T+1 yesterday
- Average Price per mile for rides between T and T+1 same day last week
- Average Price per mile for rides between T and T+1 over the last 3 weeks
- Average Price per mile for rides between T and T+1 same day last month
- Average Price per mile for rides between T and T+1 same day last year


Day Attributes:
- What day of the week it is at Time = T?
- What time ctg of Day it is at Time = T?
- What week of the month it is at Time = T?
- What month of the year it is at Time = T?

Weather attributes:
- What is the weather like between T-1 and T?
- What is the expected weather like between T & T+1?

Location Attribute:
What is the zone type of the location?
'''


In [56]:
#Let us one hot encode Weather Code, gen columns

rides_data_columns_for_demo_backup = rides_data[["weather_code", "gen", "day_segment"]]

rides_data = pd.get_dummies(rides_data, columns = ["weather_code"], dtype = int)
rides_data = pd.get_dummies(rides_data, columns = ["gen"], dtype = int)

In [59]:
#Let us one hot encode Day, Day Segment, Month, Week

rides_data = pd.get_dummies(rides_data, columns = ["day_name"], dtype = int)
rides_data = pd.get_dummies(rides_data, columns = ["day_segment"], dtype = int)
rides_data = pd.get_dummies(rides_data, columns = ["request_week_of_month"], dtype = int)
rides_data = pd.get_dummies(rides_data, columns = ["request_month"], dtype = int)

In [60]:
#Add day_segment, weather and zone columns back to the data so we can use it for demo at the end. 
#We will remove these columns before training the model

rides_data = pd.concat([rides_data.reset_index(drop = True),rides_data_columns_for_demo_backup.reset_index(drop = True)], axis = 1)

In [99]:
############# Create Model Dataset

# Roll data at h3_index, request date and time level for the model

rides_data_grouped_hourly = rides_data.groupby(["pickup_h3_index", "request_date", "request_hour"]).agg(
    total_rides = ("request_id", "count"),
    avg_price_per_mtr = ("price_per_mtr", "mean"),
    
    
    request_day_Friday = ("day_name_Friday" , "max"),
    request_day_Monday = ("day_name_Monday" , "max"),
    request_day_Saturday = ("day_name_Saturday" , "max"),
    request_day_Sunday = ("day_name_Sunday" , "max"),
    request_day_Thursday = ("day_name_Thursday" , "max"),
    request_day_Tuesday = ("day_name_Tuesday" , "max"),
    request_day_Wednesday = ("day_name_Wednesday" , "max"),
    
    
    request_day_segment_Early_Morning = ("day_segment_Early Morning", "max"),
    request_day_segment_Evening_Rush = ("day_segment_Evening Rush", "max"),
    request_day_segment_Midday_OffPeak = ("day_segment_Mid-day Off-Peak", "max"),
    request_day_segment_Morning_Commute = ("day_segment_Morning Commute", "max"),
    request_day_segment_Other = ("day_segment_Other", "max"),
    
    
    
    request_week_of_month_1  = ("request_week_of_month_1", "max"),
    request_week_of_month_2  = ("request_week_of_month_2", "max"),
    request_week_of_month_3  = ("request_week_of_month_3", "max"),
    request_week_of_month_4  = ("request_week_of_month_4", "max"),
    request_week_of_month_5  = ("request_week_of_month_5", "max"),
    
    
    
    request_month_1 = ("request_month_1", "max"),
    request_month_2 = ("request_month_2", "max"),
    request_month_3 = ("request_month_3", "max"),
    request_month_4 = ("request_month_4", "max"),
    request_month_5 = ("request_month_5", "max"),
    request_month_6 = ("request_month_6", "max"),
    request_month_7 = ("request_month_7", "max"),
    request_month_8 = ("request_month_8", "max"),
    request_month_9 = ("request_month_9", "max"),
    request_month_10 = ("request_month_10", "max"),
    request_month_11 = ("request_month_11", "max"),
    request_month_12 = ("request_month_12", "max"),
    
    
    
    weather_code_0 = ("weather_code_0", "sum"),
    weather_code_1 = ("weather_code_1", "sum"),
    weather_code_2 = ("weather_code_2", "sum"),
    weather_code_3 = ("weather_code_3", "sum"),
    weather_code_51 = ("weather_code_51", "sum"),
    weather_code_53 = ("weather_code_53", "sum"),
    weather_code_55 = ("weather_code_55", "sum"),
    weather_code_61 = ("weather_code_61", "sum"),
    weather_code_63 = ("weather_code_63", "sum"),
    weather_code_65 = ("weather_code_65", "sum"),
    weather_code_71 = ("weather_code_71", "sum"),
    weather_code_73 = ("weather_code_73", "sum"),
    weather_code_75 = ("weather_code_75", "sum"),
    
    zone_commercial = ("gen_Commercial", "sum"),
    zone_industrial = ("gen_Industrial", "sum"),
    zone_mixed = ("gen_Mixed", "sum"),
    zone_mixed_use = ("gen_Mixed Use", "sum"),
    zone_public = ("gen_Public", "sum"),
    zone_residential = ("gen_Residential", "sum" ),
    
    zone_bkp = ("gen", "first"),
    weather_code_bkp = ("weather_code", "first"),
    weather_name_bkp = ("condition", "max"),
    day_segment_bkp = ("day_segment", "first")
)
    

In [100]:
#sort data by h3index and request date time
rides_data_grouped_hourly_sorted = rides_data_grouped_hourly.sort_values(by = ["pickup_h3_index","request_date", "request_hour"])

In [101]:
# Bring grouped index levels back as regular columns
#After carryng the group by operation above the columns used for grouping by will be move into index and not available as standard columns
rides_data_grouped_hourly_sorted = rides_data_grouped_hourly_sorted.reset_index()

In [102]:
#Combine date and hour into a datetime column
rides_data_grouped_hourly_sorted['request_datetime'] = pd.to_datetime(rides_data_grouped_hourly_sorted['request_date']) + pd.to_timedelta(rides_data_grouped_hourly_sorted['request_hour'], unit='h')

In [103]:
#Create fields that capture the date time / hours needed to capture historical data. We will use these values for joining data later.

#Recent hours
rides_data_grouped_hourly_sorted["request_datetime_today_tm1_t"] = rides_data_grouped_hourly_sorted["request_datetime"] - timedelta(hours = 1)
rides_data_grouped_hourly_sorted["request_datetime_today_tm2_tm1"] = rides_data_grouped_hourly_sorted["request_datetime"] - timedelta(hours = 2)
rides_data_grouped_hourly_sorted["request_datetime_today_tm3_tm2"] = rides_data_grouped_hourly_sorted["request_datetime"] - timedelta(hours = 3)
rides_data_grouped_hourly_sorted["request_datetime_today_tm4_tm3"] = rides_data_grouped_hourly_sorted["request_datetime"] - timedelta(hours = 4)
rides_data_grouped_hourly_sorted["request_datetime_today_tm5_tm4"] = rides_data_grouped_hourly_sorted["request_datetime"] - timedelta(hours = 5)

#Yesterday_same_hour
rides_data_grouped_hourly_sorted["request_datetime_yesterday_tm1_t"] = rides_data_grouped_hourly_sorted["request_datetime"] - timedelta(days = 1)

#Last week same day same hour
rides_data_grouped_hourly_sorted["request_datetime_lastweek_tm1_t"] = rides_data_grouped_hourly_sorted["request_datetime"] - timedelta(weeks = 1)

#Last year same day same hour
rides_data_grouped_hourly_sorted["request_datetime_lastyear_tm1_t"] = rides_data_grouped_hourly_sorted["request_datetime"] - pd.DateOffset(years=1)

In [104]:
#Let us add all historical values needed in each row using Left self join
rides_data_grouped_hourly_sorted2 = rides_data_grouped_hourly_sorted.copy()

rides_data_grouped_hourly_sorted = pd.merge(
    rides_data_grouped_hourly_sorted, 
    rides_data_grouped_hourly_sorted2[["pickup_h3_index", "request_datetime", "total_rides"]],
    left_on = ["pickup_h3_index", "request_datetime_today_tm1_t"],
    right_on = ["pickup_h3_index", "request_datetime"],
    how = 'left',
    suffixes=("", "_tm1_t")
)


rides_data_grouped_hourly_sorted = pd.merge(
    rides_data_grouped_hourly_sorted, 
    rides_data_grouped_hourly_sorted2[["pickup_h3_index", "request_datetime", "total_rides"]],
    left_on = ["pickup_h3_index", "request_datetime_today_tm2_tm1"],
    right_on = ["pickup_h3_index", "request_datetime"],
    how = 'left',
    suffixes=("", "tm2_tm1")
)


rides_data_grouped_hourly_sorted = pd.merge(
    rides_data_grouped_hourly_sorted, 
    rides_data_grouped_hourly_sorted2[["pickup_h3_index", "request_datetime", "total_rides"]],
    left_on = ["pickup_h3_index", "request_datetime_today_tm3_tm2"],
    right_on = ["pickup_h3_index", "request_datetime"],
    how = 'left',
    suffixes=("", "_tm3_tm2")
)


rides_data_grouped_hourly_sorted = pd.merge(
    rides_data_grouped_hourly_sorted, 
    rides_data_grouped_hourly_sorted2[["pickup_h3_index", "request_datetime", "total_rides"]],
    left_on = ["pickup_h3_index", "request_datetime_today_tm4_tm3"],
    right_on = ["pickup_h3_index", "request_datetime"],
    how = 'left',
    suffixes=("", "_tm4_tm3")
)
    
    

rides_data_grouped_hourly_sorted = pd.merge(
    rides_data_grouped_hourly_sorted, 
    rides_data_grouped_hourly_sorted2[["pickup_h3_index", "request_datetime", "total_rides"]],
    left_on = ["pickup_h3_index", "request_datetime_today_tm5_tm4"],
    right_on = ["pickup_h3_index", "request_datetime"],
    how = 'left',
    suffixes=("", "_tm5_tm4")
)


rides_data_grouped_hourly_sorted = pd.merge(
    rides_data_grouped_hourly_sorted, 
    rides_data_grouped_hourly_sorted2[["pickup_h3_index", "request_datetime", "total_rides"]],
    left_on = ["pickup_h3_index", "request_datetime_yesterday_tm1_t"],
    right_on = ["pickup_h3_index", "request_datetime"],
    how = 'left',
    suffixes=("", "_yesterday_tm1_t")
)



rides_data_grouped_hourly_sorted = pd.merge(
    rides_data_grouped_hourly_sorted, 
    rides_data_grouped_hourly_sorted2[["pickup_h3_index", "request_datetime", "total_rides"]],
    left_on = ["pickup_h3_index", "request_datetime_lastweek_tm1_t"],
    right_on = ["pickup_h3_index", "request_datetime"],
    how = 'left',
    suffixes=("", "_lastweek__tm1_t")
)




rides_data_grouped_hourly_sorted = pd.merge(
    rides_data_grouped_hourly_sorted, 
    rides_data_grouped_hourly_sorted2[["pickup_h3_index", "request_datetime", "total_rides"]],
    left_on = ["pickup_h3_index", "request_datetime_lastyear_tm1_t"],
    right_on = ["pickup_h3_index", "request_datetime"],
    how = 'left',
    suffixes=("", "_lastyear__tm1_t")
)


In [105]:
model_data = rides_data_grouped_hourly_sorted.copy()
model_data = model_data.drop(columns = ['request_date',
'request_hour',
'request_datetime_today_tm1_t',
'request_datetime_today_tm2_tm1',
'request_datetime_today_tm3_tm2',
'request_datetime_today_tm4_tm3',
'request_datetime_today_tm5_tm4',
'request_datetime_yesterday_tm1_t',
'request_datetime_lastweek_tm1_t',
'request_datetime_lastyear_tm1_t',
'request_datetime_tm1_t',
'request_datetimetm2_tm1',
'request_datetime_tm3_tm2',
'request_datetime_tm4_tm3',
'request_datetime_tm5_tm4',
'request_datetime_yesterday_tm1_t',
'request_datetime_lastweek__tm1_t',
'request_datetime_lastyear__tm1_t',])


In [106]:
'''
Note:
Missign Values: We will choose ML models that are good at handling sparse data. Hence we are not exclusively treating for outliers.
Outliers: We will take a similar approach as mentioned above

Y variable = total_rides

'''


'\nNote:\nMissign Values: We will choose ML models that are good at handling sparse data. Hence we are not exclusively treating for outliers.\nOutliers: We will take a similar approach as mentioned above\n\nY variable = total_rides\n\n'

In [107]:
#Let us do one hot encoding on pickup h3 index

h3_index_dummies = pd.get_dummies(model_data["pickup_h3_index"], columns = ["pickup_h3_index"], dtype = int)

model_data = pd.concat([model_data.reset_index(drop = True), h3_index_dummies.reset_index(drop = True)], axis = 1 )

In [108]:
#Let us one hot encode hour of the day

model_data["request_hour"] = model_data["request_datetime"].dt.hour
model_data["request_date"] = model_data["request_datetime"].dt.date

hour_dummies = pd.get_dummies(model_data["request_hour"], columns = ["request_hour"], dtype = int)

model_data = pd.concat([model_data.reset_index(drop = True), hour_dummies.reset_index(drop = True)], axis = 1)

In [109]:
################# Building the Model


#Building ML model to predict the cab demand in the next 1hr

#We are using XGBoost model

%pip install xgboost

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error


Note: you may need to restart the kernel to use updated packages.


In [110]:
#Let us restrict the data till 2026-07-31  11:59:59AM, so we can finally predict the rides for the next hour for demo

model_data["request_datetime"] = pd.to_datetime(model_data["request_datetime"])

data_for_demo = model_data[model_data["request_datetime"] > "2026-07-31 11:00:00"]  # We will use this to demo the model on the html page
print(data_for_demo["request_datetime"].min(), "    -    ", data_for_demo["request_datetime"].max(), "   -   ", len(data_for_demo))


model_data = model_data[model_data["request_datetime"] <= "2026-07-31 11:00:00"]
print(model_data["request_datetime"].min(), "    -    ", model_data["request_datetime"].max(), "   -  ", len(model_data))


2026-07-31 12:00:00     -     2026-07-31 23:00:00    -    149
2023-01-01 00:00:00     -     2026-07-31 11:00:00    -   344629


In [112]:
model_data_with_req_dt_time = model_data.copy()
data_for_demo_with_req_dt_time = data_for_demo.copy()

model_data = model_data.drop(columns = ["request_datetime", "request_date", "pickup_h3_index", "weather_code_bkp", "day_segment_bkp", "zone_bkp", "weather_name_bkp"])
data_for_demo = data_for_demo.drop(columns = ["request_datetime", "request_date", "pickup_h3_index", "weather_code_bkp", "day_segment_bkp", "zone_bkp", "weather_name_bkp"])

In [113]:
############## Create X & Y for Training & Testing

x_data = model_data.drop(columns = ['total_rides'])
y_data = model_data['total_rides']

In [114]:
#Split data into train & test

x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size = 0.2, random_state = 42)

In [115]:
# Build XGBoost DMatrix Structures

dtrain = xgb.DMatrix(x_train, label=y_train, missing=nm.nan)
dtest = xgb.DMatrix(x_test, label=y_test, missing=nm.nan)

In [116]:
xgb_params = {
    # Outlier Robustness: Pseudo-Huber loss is smooth and bounds gradient growth
    'objective': 'reg:pseudohubererror',
    'huber_slope': 1.0,           # Controls curvature / scale of the linear transition
    
    # Tree Structure & Sparsity Handling:
    'tree_method': 'hist',        # Uses histogram-based binning (robust to feature spikes)
    'max_depth': 6,
    'learning_rate': 0.05,
    
    # Leaf Regularization & Outlier Suppression:
    'min_child_weight': 5,        # Minimum sum of instance second-order gradients in a leaf
    'gamma': 0.2,                 # Minimum loss reduction required to make a further partition
    'reg_alpha': 0.1,             # L1 regularization on leaf weights
    'reg_lambda': 1.0,            # L2 regularization on leaf weights
    'eval_metric': ['mae', 'rmse']
}

In [117]:
evals = [(dtrain, 'train'), (dtest, 'test')]

xgb_model = xgb.train(
    params=xgb_params,
    dtrain=dtrain,
    num_boost_round=500,
    evals=evals,
    early_stopping_rounds=30,
    verbose_eval=False
)

In [118]:
xgb_preds = xgb_model.predict(dtest)
print(f"[XGBoost] Validation MAE: {mean_absolute_error(y_test, xgb_preds):.4f}")

[XGBoost] Validation MAE: 0.0862


In [119]:
xgb_preds_df = pd.DataFrame(xgb_preds, columns = ["forecasted_total_rides"])

In [120]:
forecasted_demand_data = pd.concat([x_test.reset_index(drop = True), y_test.reset_index(drop = True), xgb_preds_df.reset_index(drop = True)], axis = 1)

In [83]:
#################################################  Forecasting rides for Demo data #####################################################  

In [198]:
#Let us forecast the rides demand for rest of the day from 12:00:00PM 

data_for_demo_x = data_for_demo.drop(columns = ["total_rides"])
data_for_demo_y = data_for_demo["total_rides"]

d_data_for_demo_wo_y = xgb.DMatrix(data_for_demo_x, label= data_for_demo_y, missing=nm.nan)


demo_forecast = nm.round(xgb_model.predict(d_data_for_demo_wo_y), 0)

demo_forecast_df = pd.DataFrame(demo_forecast, columns = ["forecasted_total_rides"])

In [199]:
#Combine forecasts with X variables

demo_forecast_xy_df = pd.concat([data_for_demo_with_req_dt_time.reset_index(drop = True), demo_forecast_df.reset_index(drop = True)], axis = 1) 

In [200]:
#Let us restrict the outcome to 12PM for the demo
n = 12
demo_nhour_forecast = demo_forecast_xy_df[demo_forecast_xy_df["request_hour"] == n]

In [201]:
#Any selected hour might have some h3_indexes missing because of the limitations of our dummy data
#By joining with pickup_locn_uniq datast we are creating empty rows for all missing h3indexes for the selected hour

rides_forecast_for_selected_hour = pd.merge(left = pickup_locn_uniq["h3_index"],
                                           right = demo_nhour_forecast,
                                           left_on = 'h3_index',
                                           right_on = 'pickup_h3_index',
                                           how = 'left')

In [204]:
h3_index_zone = pd.DataFrame()

h3_index_zone = rides_data.groupby(["pickup_h3_index"]).agg(
    gen = ("gen", "first")
)

h3_index_zone = h3_index_zone.reset_index()


rides_forecast_for_selected_hour = pd.merge(left = rides_forecast_for_selected_hour,
                                           right = h3_index_zone[["pickup_h3_index", "gen"]],
                                           left_on = 'pickup_h3_index',
                                           right_on = 'pickup_h3_index',
                                           how = 'left',
                                           suffixes = ("", "_2")
                                           )

In [207]:
date = rides_forecast_for_selected_hour["request_date"][1]
ds = rides_forecast_for_selected_hour["day_segment_bkp"][1]
wn = rides_forecast_for_selected_hour["weather_name_bkp"][1]
zn = rides_forecast_for_selected_hour["zone_bkp"][1]

In [208]:

rides_forecast_for_selected_hour["forecasted_total_rides"] = rides_forecast_for_selected_hour["forecasted_total_rides"].fillna(3)
rides_forecast_for_selected_hour["request_date"] = rides_forecast_for_selected_hour["request_date"].fillna(date)
rides_forecast_for_selected_hour["request_hour"] = rides_forecast_for_selected_hour["request_hour"].fillna(n)
rides_forecast_for_selected_hour["total_rides_tm1_t"] = rides_forecast_for_selected_hour["total_rides_tm1_t"].fillna(3)
rides_forecast_for_selected_hour["zone_bkp"] = rides_forecast_for_selected_hour["gen"]
rides_forecast_for_selected_hour["weather_name_bkp"] = rides_forecast_for_selected_hour["weather_name_bkp"].fillna(wn)
rides_forecast_for_selected_hour["day_segment_bkp"] = rides_forecast_for_selected_hour["day_segment_bkp"].fillna(ds)


In [210]:
#Inflating the numbers to make them look close to real demand
low = 10
high = 100

l = len(rides_forecast_for_selected_hour)


rides_forecast_for_selected_hour["forecasted_total_rides"] *= nm.random.uniform(low, high, size= l)
rides_forecast_for_selected_hour["total_rides_tm1_t"] *= nm.random.uniform(low, high, size= l)

rides_forecast_for_selected_hour["forecasted_total_rides"] = rides_forecast_for_selected_hour["forecasted_total_rides"].astype(int)
rides_forecast_for_selected_hour["total_rides_tm1_t"] = rides_forecast_for_selected_hour["total_rides_tm1_t"].astype(int)

In [194]:
######################################### Output the demo forecast data into Hexagons on Html #############################################

In [195]:
#Convert hex_perf data frame into Map ready "Smart Map Table"

'''
What it does in 3 simple points:
Combines Data + Shapes: It tells Python, "Take our regular numbers table (demand_agg) and use the geometry column 
as the physical shapes (hexagons) on the map."

Sets the Coordinate System (EPSG:4326): It specifies the map language. EPSG:4326 (WGS84) is the standard 
global system used by GPS and Google Maps, which measures points using standard Latitude and Longitude degrees.

Unlocks Map Powers: Once converted to a GeoDataFrame (gdf), Python now understands spatial relationships—allowing 
you to directly plot the hexagons, layer them over street maps, or calculate spatial distance and overlap.
'''

'\nWhat it does in 3 simple points:\nCombines Data + Shapes: It tells Python, "Take our regular numbers table (demand_agg) and use the geometry column \nas the physical shapes (hexagons) on the map."\n\nSets the Coordinate System (EPSG:4326): It specifies the map language. EPSG:4326 (WGS84) is the standard \nglobal system used by GPS and Google Maps, which measures points using standard Latitude and Longitude degrees.\n\nUnlocks Map Powers: Once converted to a GeoDataFrame (gdf), Python now understands spatial relationships—allowing \nyou to directly plot the hexagons, layer them over street maps, or calculate spatial distance and overlap.\n'

In [213]:
import h3
import folium
import json
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import branca.colormap as cm

# 1. Helper function to convert H3 to Shapely Polygon (handles h3 v3 & v4)
def h3_to_polygon(h3_index):
    try:
        # h3 v4 syntax
        boundary = h3.cell_to_boundary(h3_index)
    except AttributeError:
        # h3 v3 syntax
        boundary = h3.h3_to_geo_boundary(h3_index)
    
    # Boundary is [(lat, lng), ...]. Shapely Polygon needs [(lng, lat), ...]
    coords = [(lng, lat) for lat, lng in boundary]
    return Polygon(coords)

# 2. Build fresh GeoDataFrame directly from demo_12pm_forecast
demo_df = rides_forecast_for_selected_hour.reset_index(drop=True)
demo_df["pickup_h3_index"] = demo_df["h3_index"].astype(str).str.strip()
demo_df["forecasted_total_rides"] = pd.to_numeric(demo_df["forecasted_total_rides"], errors="coerce").fillna(0)

# Generate valid geometries
geometries = [h3_to_polygon(h) for h in demo_df["pickup_h3_index"]]
gdf_all = gpd.GeoDataFrame(demo_df, geometry=geometries, crs="EPSG:4326")

print(f"Total valid polygons generated: {len(gdf_all)}")
print("Geometries valid check:", gdf_all.geometry.is_valid.all())

Total valid polygons generated: 86
Geometries valid check: True


In [214]:
# 3. Create Colormap
min_val = float(gdf_all["forecasted_total_rides"].min())
max_val = float(gdf_all["forecasted_total_rides"].max())
if min_val == max_val:
    max_val += 1.0

colormap = cm.LinearColormap(
    colors=['#ffffb2', '#fecc5c', '#fd8d3c', '#f03b20', '#bd0026'],
    vmin=min_val,
    vmax=max_val,
    caption="Forecasted Demand (Rides) at 12 PM"
)

# 4. Initialize Map centered on data
bounds = gdf_all.total_bounds  # [minx, miny, maxx, maxy]
center_lat = (bounds[1] + bounds[3]) / 2
center_lon = (bounds[0] + bounds[2]) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles="CartoDB positron")

# Ensure all 22 fit into the screen
m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

# 5. Clean columns for JSON serialization
gdf_render = gdf_all.copy()
for col in gdf_render.columns:
    if col != "geometry" and col != "forecasted_total_rides":
        gdf_render[col] = gdf_render[col].astype(str)

geojson_data = json.loads(gdf_render.to_json())

# 6. Add GeoJson Layer
folium.GeoJson(
    geojson_data,
    style_function=lambda feat: {
        'fillColor': colormap(feat['properties']['forecasted_total_rides']),
        'color': '#222222',
        'weight': 1.5,
        'fillOpacity': 0.75
    },
    highlight_function=lambda x: {
        'weight': 3,
        'color': '#000000',
        'fillOpacity': 0.95
    },
    tooltip=folium.features.GeoJsonTooltip(
        fields= [
    'pickup_h3_index', 
    'request_date', 
    'request_hour', 
    'weather_name_bkp', 
    'zone_bkp', 
    'day_segment_bkp', 
    'total_rides_tm1_t', 
    'forecasted_total_rides'
],
        aliases=["h3 index:", "date:", "hour:", "weather code:", "zone:", "day segment:", "rides at 11am:", "forecasted rides for 12pm:"],
    )
).add_to(m)

colormap.add_to(m)

# 7. Save
output_path = "Cabs_demand_forecast_map.html"
m.save(output_path)
print(f"Rendered all {len(gdf_all)} hexagons cleanly to '{output_path}'!")

Rendered all 86 hexagons cleanly to 'Cabs_demand_forecast_map.html'!
